In [46]:
import sys
import torch
import os
# sys.path.append('/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/dpvo')
import dpvo
import collections
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np
verbose = False


In [47]:
# load onnx models
so = ort.SessionOptions()
so.log_severity_level = 2  # 0 = verbose

onnx_dir = '/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/andy/onnx'
print(f'Loading onnx update modular model')
module_paths = {'corr': os.path.join(onnx_dir, "corr.onnx"),
                'norm': os.path.join(onnx_dir, "norm.onnx"),
                'c1': os.path.join(onnx_dir, "c1.onnx"),
                'c2': os.path.join(onnx_dir, "c2.onnx"),
                'agg_kk': os.path.join(onnx_dir, "agg_kk.onnx"),
                'agg_ij': os.path.join(onnx_dir, "agg_ij.onnx"),
                'gru': os.path.join(onnx_dir, "gru.onnx"),
                'w': os.path.join(onnx_dir, "w.onnx"),
                'd': os.path.join(onnx_dir, "d.onnx")}
module_sessions = {}
for name, module_path in module_paths.items():
    if not os.path.isfile(module_path):
        raise FileNotFoundError(f"ONNX encoder file not found in {onnx_dir} for {name}. Run andy/onnx_conversion.ipynb first.")
    onnx_dir_str = os.path.normpath(str(onnx_dir))
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    # providers = ["CUDAExecutionProvider"]
    module_sessions[name] = ort.InferenceSession(module_path, sess_options=so, providers=providers)
    if verbose:
        print("=== inputs ===")
        for i in module_sessions[name].get_inputs():
            print(i.name, i.shape, i.type)
        print("=== outputs ===")
        for o in module_sessions[name].get_outputs():
            print(o.name, o.shape, o.type)
    if verbose: print(f'Onnx {name} module loaded: {module_sessions[name]}')

Loading onnx update modular model


In [48]:
# load pytorch models
project_root = '/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/'

from dpvo.net import VONet

# --- 1. Load the DPVO PyTorch Model ---
# Weights path: project root dpvo.pth or andy/dpvo.pth
pth_model_path = os.path.join(project_root, "dpvo.pth")
if not os.path.isfile(pth_model_path):
    pth_model_path = os.path.join(project_root, "andy", "dpvo.pth")
assert os.path.isfile(pth_model_path), f"Checkpoint not found: {pth_model_path}"

export_device = torch.device("cuda")

model = VONet()

ckpt = torch.load(pth_model_path, map_location="cuda", weights_only=True)
state_dict = ckpt if isinstance(ckpt, dict) and "state_dict" not in ckpt else ckpt.get("state_dict", ckpt)

# Strip DataParallel prefix; drop update.lmbda if present (DPVO load_weights does this)
new_state_dict = collections.OrderedDict()
for k, v in state_dict.items():
    if "update.lmbda" in k:
        print(k)
        continue
    new_state_dict[k.replace("module.", "")] = v
    print(k)

missing, unexpected = model.load_state_dict(new_state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval().to(export_device)



module.patchify.fnet.conv1.weight
module.patchify.fnet.conv1.bias
module.patchify.fnet.layer1.0.conv1.weight
module.patchify.fnet.layer1.0.conv1.bias
module.patchify.fnet.layer1.0.conv2.weight
module.patchify.fnet.layer1.0.conv2.bias
module.patchify.fnet.layer1.1.conv1.weight
module.patchify.fnet.layer1.1.conv1.bias
module.patchify.fnet.layer1.1.conv2.weight
module.patchify.fnet.layer1.1.conv2.bias
module.patchify.fnet.layer2.0.conv1.weight
module.patchify.fnet.layer2.0.conv1.bias
module.patchify.fnet.layer2.0.conv2.weight
module.patchify.fnet.layer2.0.conv2.bias
module.patchify.fnet.layer2.0.downsample.0.weight
module.patchify.fnet.layer2.0.downsample.0.bias
module.patchify.fnet.layer2.1.conv1.weight
module.patchify.fnet.layer2.1.conv1.bias
module.patchify.fnet.layer2.1.conv2.weight
module.patchify.fnet.layer2.1.conv2.bias
module.patchify.fnet.conv2.weight
module.patchify.fnet.conv2.bias
module.patchify.inet.conv1.weight
module.patchify.inet.conv1.bias
module.patchify.inet.layer1.0.co

VONet(
  (patchify): Patchifier(
    (fnet): BasicEncoder4(
      (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (conv1): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
      (relu1): ReLU(inplace=True)
      (layer1): Sequential(
        (0): ResidualBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (relu): ReLU(inplace=True)
          (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        )
        (1): ResidualBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (relu): ReLU(inplace=True)
          

In [49]:
# load dummy_inputs:
INPUT_PAYLOAD_PATH = os.path.join('onnx', 'input_payload.pth')

device = 'cuda'

payload = torch.load(INPUT_PAYLOAD_PATH, map_location=device)
net = payload['net_in'].float()
ctx = payload['inp'].float()
corr = payload['corr'].float()
ii = payload['ii'].long()
jj = payload['jj'].long()
kk = payload['kk'].long()
E_real = int(net.shape[1])

B, E_real, D = net.shape
_, _, Cc = corr.shape


/tmp/ipykernel_3308518/3654949096.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(INPUT_PAYLOAD_PATH, map_location=device)


In [50]:
# helper methods
def bind_torch_inputs(io_binding, inputs: dict, device="cuda", device_id=0):
    for name, tensor in inputs.items():
        if tensor is None:
            continue

        assert tensor.is_cuda, f"{name} must be on GPU for zero-copy"

        io_binding.bind_input(
            name=name,
            device_type=device,
            device_id=device_id,
            element_type=(
                np.float32 if tensor.dtype in (torch.float32, torch.float16)
                else np.int64
            ),
            shape=tuple(tensor.shape),
            buffer_ptr=tensor.data_ptr(),
        )

def assert_close(name, a, b, atol=1e-8, rtol=1e-5):
    d = (a - b).abs().max().item()
    ok = torch.allclose(a, b, atol=atol, rtol=rtol)
    print(f"{name}: max_abs_diff={d:.3e}  allclose={ok}")
    # assert ok, f"{name} mismatch"
    

In [51]:
#--------------------corr----------------------

corr_t = corr.to(torch.float32, copy=False).contiguous()

# -----------ONNX----------------------
corr_onnx_out = torch.empty((B, E_real, D), device=corr_t.device, dtype=torch.float32)
corr_io_binding = module_sessions['corr'].io_binding()
bind_torch_inputs(corr_io_binding, {'corr_input': corr_t})

corr_io_binding.bind_output(
    name='corr_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(corr_onnx_out.shape),
    buffer_ptr=corr_onnx_out.data_ptr(),
)
module_sessions['corr'].run_with_iobinding(corr_io_binding)

print(f'Onnx Output: {corr_onnx_out}')

# -----------Pytorch----------------------
corr_pytorch_out = model.update.corr(corr_t)

print(f'Onnx Output: {corr_pytorch_out}')

# -----------Comparison---------------------
assert_close("corr", corr_onnx_out, corr_pytorch_out)


Onnx Output: tensor([[[-1.3759,  0.2102,  1.6980,  ...,  1.0415, -0.3800,  1.9120],
         [-1.1505, -0.1133,  1.2951,  ...,  0.9306, -0.6190,  1.7636],
         [-1.5378,  0.1334,  2.0349,  ...,  0.8870, -0.6580,  1.3073],
         ...,
         [-0.3976,  1.4376, -0.1109,  ..., -0.3681,  1.6351,  1.4819],
         [-0.5344,  1.4767,  0.0074,  ..., -0.4587,  1.5549,  1.1205],
         [-0.6370,  1.4841,  0.0676,  ..., -0.5390,  1.4941,  1.0154]]],
       device='cuda:0')
Onnx Output: tensor([[[-1.3760,  0.2102,  1.6987,  ...,  1.0417, -0.3798,  1.9119],
         [-1.1504, -0.1133,  1.2960,  ...,  0.9309, -0.6186,  1.7634],
         [-1.5382,  0.1328,  2.0352,  ...,  0.8873, -0.6580,  1.3066],
         ...,
         [-0.3974,  1.4386, -0.1100,  ..., -0.3677,  1.6355,  1.4810],
         [-0.5343,  1.4769,  0.0084,  ..., -0.4592,  1.5548,  1.1194],
         [-0.6370,  1.4837,  0.0686,  ..., -0.5392,  1.4946,  1.0153]]],
       device='cuda:0', grad_fn=<ViewBackward0>)
corr: max_abs_dif

In [52]:
#--------------------norm----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

# -----------ONNX----------------------
norm_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
norm_io_binding = module_sessions['norm'].io_binding()
bind_torch_inputs(norm_io_binding, {'net_input': net_t})

norm_io_binding.bind_output(
    name='norm_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(norm_onnx_out.shape),
    buffer_ptr=norm_onnx_out.data_ptr(),
)
module_sessions['norm'].run_with_iobinding(norm_io_binding)

print(f'Onnx Output: {norm_onnx_out}')

# -----------Pytorch----------------------
norm_pytorch_out = model.update.norm(net_t)

print(f'Onnx Output: {norm_pytorch_out}')

# -----------Comparison---------------------
assert_close("norm", norm_onnx_out, norm_pytorch_out)


Onnx Output: tensor([[[ 0.1278,  0.2158, -0.0417,  ..., -0.5236, -1.2212,  0.0847],
         [ 0.2564,  0.1029, -0.0902,  ..., -0.6847, -1.5460, -0.1939],
         [ 0.2689,  0.1163,  0.0229,  ..., -0.5182, -1.1262,  0.1860],
         ...,
         [ 0.5324,  0.2949,  0.1510,  ...,  0.2632, -0.9921,  0.2526],
         [ 0.6044,  0.2637, -0.0741,  ...,  0.3061, -0.8952,  0.1582],
         [ 0.4174,  0.2708, -0.3029,  ...,  0.1258, -0.8824,  0.4536]]],
       device='cuda:0')
Onnx Output: tensor([[[ 0.1278,  0.2158, -0.0417,  ..., -0.5236, -1.2212,  0.0847],
         [ 0.2564,  0.1029, -0.0902,  ..., -0.6847, -1.5460, -0.1939],
         [ 0.2689,  0.1163,  0.0229,  ..., -0.5182, -1.1262,  0.1860],
         ...,
         [ 0.5324,  0.2949,  0.1510,  ...,  0.2632, -0.9921,  0.2526],
         [ 0.6044,  0.2637, -0.0741,  ...,  0.3061, -0.8952,  0.1582],
         [ 0.4174,  0.2708, -0.3029,  ...,  0.1258, -0.8824,  0.4536]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>)
norm: 

In [53]:
#--------------------c1----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

# -----------ONNX----------------------
c1_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
c1_io_binding = module_sessions['c1'].io_binding()
bind_torch_inputs(c1_io_binding, {'c1_input': net_t})

c1_io_binding.bind_output(
    name='c1_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(c1_onnx_out.shape),
    buffer_ptr=c1_onnx_out.data_ptr(),
)
module_sessions['c1'].run_with_iobinding(c1_io_binding)

print(f'Onnx Output: {c1_onnx_out}')

# -----------Pytorch----------------------
c1_pytorch_out = model.update.c1(net_t)

print(f'Onnx Output: {c1_pytorch_out}')

# -----------Comparison---------------------
assert_close("c1", c1_onnx_out, c1_pytorch_out)


Onnx Output: tensor([[[ 7.6211,  0.1458,  4.6569,  ...,  8.6765,  4.5624, -2.4695],
         [ 7.4968, -0.7892,  6.7043,  ...,  8.2513,  4.2120, -0.2688],
         [ 8.8562, -0.0210,  5.1722,  ...,  9.2554,  4.3180, -2.3467],
         ...,
         [ 8.5053,  2.0957,  4.8564,  ...,  7.3910,  5.6701, -2.7638],
         [ 8.7628,  1.2061,  5.4674,  ...,  6.2234,  5.2130, -2.5517],
         [10.1305,  1.3389,  6.6215,  ...,  5.4780,  4.4254, -2.4944]]],
       device='cuda:0')
Onnx Output: tensor([[[ 7.6212,  0.1439,  4.6550,  ...,  8.6775,  4.5623, -2.4704],
         [ 7.4970, -0.7890,  6.7025,  ...,  8.2535,  4.2116, -0.2694],
         [ 8.8570, -0.0213,  5.1714,  ...,  9.2569,  4.3169, -2.3481],
         ...,
         [ 8.5065,  2.0946,  4.8548,  ...,  7.3916,  5.6681, -2.7643],
         [ 8.7627,  1.2066,  5.4683,  ...,  6.2242,  5.2124, -2.5524],
         [10.1288,  1.3378,  6.6203,  ...,  5.4785,  4.4233, -2.4956]]],
       device='cuda:0', grad_fn=<ViewBackward0>)
c1: max_abs_diff=

In [54]:
#--------------------c2----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

# -----------ONNX----------------------
c2_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
c2_io_binding = module_sessions['c2'].io_binding()
bind_torch_inputs(c2_io_binding, {'c2_input': net_t})

c2_io_binding.bind_output(
    name='c2_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(c2_onnx_out.shape),
    buffer_ptr=c2_onnx_out.data_ptr(),
)
module_sessions['c2'].run_with_iobinding(c2_io_binding)

print(f'Onnx Output: {c2_onnx_out}')

# -----------Pytorch----------------------
c2_pytorch_out = model.update.c2(net_t)

print(f'Onnx Output: {c2_pytorch_out}')

# -----------Comparison---------------------
assert_close("c2", c2_onnx_out, c2_pytorch_out)


Onnx Output: tensor([[[ 2.3359, -1.3912,  6.2403,  ..., -1.8425,  0.1223,  3.3622],
         [ 2.8836, -1.2500,  6.2488,  ..., -0.3891,  0.8497,  3.2935],
         [ 2.4878, -1.7489,  6.9969,  ..., -1.9720, -0.1309,  3.7805],
         ...,
         [ 2.1441, -0.7608,  6.3107,  ..., -2.0373, -0.7280,  3.6266],
         [ 2.2205, -0.2550,  6.1797,  ..., -2.0915, -0.7271,  4.8748],
         [ 1.6307, -0.0685,  6.3222,  ..., -2.4192, -0.1885,  4.3627]]],
       device='cuda:0')
Onnx Output: tensor([[[ 2.3361, -1.3904,  6.2397,  ..., -1.8405,  0.1226,  3.3630],
         [ 2.8836, -1.2499,  6.2493,  ..., -0.3892,  0.8488,  3.2928],
         [ 2.4877, -1.7478,  6.9981,  ..., -1.9706, -0.1303,  3.7795],
         ...,
         [ 2.1444, -0.7605,  6.3103,  ..., -2.0363, -0.7284,  3.6260],
         [ 2.2210, -0.2543,  6.1807,  ..., -2.0902, -0.7280,  4.8745],
         [ 1.6303, -0.0678,  6.3220,  ..., -2.4177, -0.1877,  4.3626]]],
       device='cuda:0', grad_fn=<ViewBackward0>)
c2: max_abs_diff=

In [55]:
#--------------------agg_kk----------------------

net_t = net.to(torch.float32, copy=False).contiguous()
_, jx_kk = torch.unique(kk, return_inverse=True)

# -----------ONNX----------------------
agg_kk_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
agg_kk_io_binding = module_sessions['agg_kk'].io_binding()
bind_torch_inputs(agg_kk_io_binding, {'agg_kk_input': net_t, 'jx_input': jx_kk})

agg_kk_io_binding.bind_output(
    name='agg_kk_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(agg_kk_onnx_out.shape),
    buffer_ptr=agg_kk_onnx_out.data_ptr(),
)
module_sessions['agg_kk'].run_with_iobinding(agg_kk_io_binding)

print(f'Onnx Output: {agg_kk_onnx_out}')

# -----------Pytorch----------------------
agg_kk_pytorch_out = model.update.agg_kk(net_t, jx_kk)

print(f'Onnx Output: {agg_kk_pytorch_out}')

# -----------Comparison---------------------
assert_close("agg_kk", agg_kk_onnx_out, agg_kk_pytorch_out)


Onnx Output: tensor([[[ 1.7797,  0.5459,  1.7666,  ...,  0.1486, -0.0279,  2.0223],
         [ 1.7797,  0.5459,  1.7666,  ...,  0.1486, -0.0279,  2.0223],
         [ 1.7797,  0.5459,  1.7666,  ...,  0.1486, -0.0279,  2.0223],
         ...,
         [ 1.1945,  1.3503,  1.9770,  ...,  0.1766, -0.7575,  1.4769],
         [ 1.1945,  1.3503,  1.9770,  ...,  0.1766, -0.7575,  1.4769],
         [ 1.1945,  1.3503,  1.9770,  ...,  0.1766, -0.7575,  1.4769]]],
       device='cuda:0')
Onnx Output: tensor([[[ 1.7802,  0.5454,  1.7671,  ...,  0.1484, -0.0291,  2.0215],
         [ 1.7802,  0.5454,  1.7671,  ...,  0.1484, -0.0291,  2.0215],
         [ 1.7802,  0.5454,  1.7671,  ...,  0.1484, -0.0291,  2.0215],
         ...,
         [ 1.1955,  1.3495,  1.9768,  ...,  0.1771, -0.7577,  1.4764],
         [ 1.1955,  1.3495,  1.9768,  ...,  0.1771, -0.7577,  1.4764],
         [ 1.1955,  1.3495,  1.9768,  ...,  0.1771, -0.7577,  1.4764]]],
       device='cuda:0', grad_fn=<IndexBackward0>)
agg_kk: max_abs_

2026-03-25 16:59:11.214067128 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0


In [56]:
#--------------------agg_ij----------------------

net_t = net.to(torch.float32, copy=False).contiguous()
iijj = ii * 12345 + jj
_, jx_ij = torch.unique(iijj, return_inverse=True)

# -----------ONNX----------------------
agg_ij_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
agg_ij_io_binding = module_sessions['agg_ij'].io_binding()
bind_torch_inputs(agg_ij_io_binding, {'agg_ij_input': net_t, 'jx_input': jx_ij})

agg_ij_io_binding.bind_output(
    name='agg_ij_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(agg_ij_onnx_out.shape),
    buffer_ptr=agg_ij_onnx_out.data_ptr(),
)
module_sessions['agg_ij'].run_with_iobinding(agg_ij_io_binding)

print(f'Onnx Output: {agg_ij_onnx_out}')

# -----------Pytorch----------------------
agg_ij_pytorch_out = model.update.agg_ij(net_t, jx_ij)

print(f'Onnx Output: {agg_ij_pytorch_out}')

# -----------Comparison---------------------
assert_close("agg_ij", agg_ij_onnx_out, agg_ij_pytorch_out)


Onnx Output: tensor([[[-7.7664e-01,  1.8179e-01, -8.2485e-02,  ...,  3.2399e+00,
          -4.5084e-01, -1.6120e+00],
         [-6.9376e-01,  2.9520e-02, -4.6209e-01,  ...,  3.1137e+00,
          -5.7780e-01, -1.8850e+00],
         [-8.9246e-01,  3.2739e-03, -1.7674e-01,  ...,  2.8847e+00,
          -5.9292e-01, -1.6772e+00],
         ...,
         [ 1.3317e+00, -1.6990e+00,  1.4666e+00,  ...,  2.3559e+00,
          -1.9058e+00, -3.2205e+00],
         [ 1.7116e+00, -1.6267e+00,  1.9588e+00,  ...,  1.9416e+00,
          -1.8558e+00, -3.6039e+00],
         [ 1.6106e+00, -1.7312e+00,  2.1598e+00,  ...,  1.7703e+00,
          -2.1334e+00, -3.6793e+00]]], device='cuda:0')
Onnx Output: tensor([[[-7.7638e-01,  1.8286e-01, -8.2233e-02,  ...,  3.2389e+00,
          -4.5118e-01, -1.6115e+00],
         [-6.9375e-01,  2.9798e-02, -4.6258e-01,  ...,  3.1133e+00,
          -5.7863e-01, -1.8860e+00],
         [-8.9200e-01,  3.6233e-03, -1.7680e-01,  ...,  2.8839e+00,
          -5.9328e-01, -1.6779e+0

2026-03-25 16:59:11.260570607 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,466,384} for output /Identity_3_output_0


In [57]:
#--------------------gru----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

# -----------ONNX----------------------
gru_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
gru_io_binding = module_sessions['gru'].io_binding()
bind_torch_inputs(gru_io_binding, {'gru_input': net_t})

gru_io_binding.bind_output(
    name='net_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(gru_onnx_out.shape),
    buffer_ptr=gru_onnx_out.data_ptr(),
)
module_sessions['gru'].run_with_iobinding(gru_io_binding)

print(f'Onnx Output: {gru_onnx_out}')

# -----------Pytorch----------------------
gru_pytorch_out = model.update.gru(net_t)

print(f'Onnx Output: {gru_pytorch_out}')

# -----------Comparison---------------------
assert_close("gru", gru_onnx_out, gru_pytorch_out)


Onnx Output: tensor([[[-0.8348, -0.5173, -0.4597,  ..., -2.5187, -2.6054,  0.0031],
         [-0.5461, -0.7382, -0.6902,  ..., -2.6078, -2.5532, -0.2979],
         [-0.7737, -0.5928, -0.4139,  ..., -2.2568, -2.0183,  0.1862],
         ...,
         [-0.5110, -0.3916, -0.4595,  ..., -2.2657, -2.1634,  0.3914],
         [-0.7488, -0.0838, -0.4148,  ..., -2.1640, -1.9237,  0.3461],
         [-1.1986,  0.2146, -0.5240,  ..., -2.1141, -1.2965,  0.5178]]],
       device='cuda:0')
Onnx Output: tensor([[[-0.8347, -0.5171, -0.4597,  ..., -2.5186, -2.6056,  0.0030],
         [-0.5462, -0.7384, -0.6903,  ..., -2.6074, -2.5527, -0.2971],
         [-0.7740, -0.5924, -0.4136,  ..., -2.2564, -2.0180,  0.1863],
         ...,
         [-0.5106, -0.3921, -0.4594,  ..., -2.2659, -2.1629,  0.3914],
         [-0.7488, -0.0838, -0.4144,  ..., -2.1634, -1.9233,  0.3460],
         [-1.1989,  0.2140, -0.5238,  ..., -2.1135, -1.2963,  0.5181]]],
       device='cuda:0', grad_fn=<AddBackward0>)
gru: max_abs_diff=

In [58]:
#--------------------w----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

# -----------ONNX----------------------
w_onnx_out = torch.empty((B, E_real, 2), device=net_t.device, dtype=torch.float32)
w_io_binding = module_sessions['w'].io_binding()
bind_torch_inputs(w_io_binding, {'net_input': net_t})

w_io_binding.bind_output(
    name='weight_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(w_onnx_out.shape),
    buffer_ptr=w_onnx_out.data_ptr(),
)
module_sessions['w'].run_with_iobinding(w_io_binding)

print(f'Onnx Output: {w_onnx_out}')

# -----------Pytorch----------------------
w_pytorch_out = model.update.w(net_t)

print(f'Onnx Output: {w_pytorch_out}')

# -----------Comparison---------------------
assert_close("w", w_onnx_out, w_pytorch_out)


Onnx Output: tensor([[[0.2102, 0.1048],
         [0.1849, 0.1014],
         [0.2296, 0.1083],
         ...,
         [0.2311, 0.1056],
         [0.3373, 0.1222],
         [0.3045, 0.1186]]], device='cuda:0')
Onnx Output: tensor([[[0.2103, 0.1048],
         [0.1849, 0.1014],
         [0.2297, 0.1083],
         ...,
         [0.2311, 0.1057],
         [0.3374, 0.1222],
         [0.3045, 0.1186]]], device='cuda:0', grad_fn=<SigmoidBackward0>)
w: max_abs_diff=2.391e-04  allclose=False


In [59]:
#--------------------d----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

# -----------ONNX----------------------
d_onnx_out = torch.empty((B, E_real, 2), device=net_t.device, dtype=torch.float32)
d_io_binding = module_sessions['d'].io_binding()
bind_torch_inputs(d_io_binding, {'net_input': net_t})

d_io_binding.bind_output(
    name='delta_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(d_onnx_out.shape),
    buffer_ptr=d_onnx_out.data_ptr(),
)
module_sessions['d'].run_with_iobinding(d_io_binding)

print(f'Onnx Output: {d_onnx_out}')

# -----------Pytorch----------------------
d_pytorch_out = model.update.d(net_t)

print(f'Onnx Output: {d_pytorch_out}')

# -----------Comparison---------------------
assert_close("d", d_onnx_out, d_pytorch_out)


Onnx Output: tensor([[[ 0.2083, -0.1113],
         [-0.2077,  0.5034],
         [ 0.1534, -0.2272],
         ...,
         [ 0.2115, -0.0031],
         [-0.0699, -0.0810],
         [-0.0756,  0.0329]]], device='cuda:0')
Onnx Output: tensor([[[ 0.2082, -0.1113],
         [-0.2078,  0.5033],
         [ 0.1533, -0.2272],
         ...,
         [ 0.2114, -0.0032],
         [-0.0699, -0.0810],
         [-0.0757,  0.0329]]], device='cuda:0', grad_fn=<GradClipBackward>)
d: max_abs_diff=9.190e-03  allclose=False
